# Multiplex immunofluorescence on Visium HD

A fourth kind of "align a technology's own image onto Visium HD" tutorial, alongside the MSI
notebooks -- this time the moving modality is a multiplex IF scan (8-channel, Opal/Akoya-style,
`IndicaLabs`-format pyramidal TIFF: panCK, C31, FAP, CD34, PDPN, aSMA, DAPI, autofluorescence)
rather than a point-grid instrument.

**Why this is simpler than the MSI pipeline, structurally**: MSI data needs two alignment steps
because its own acquisition grid doesn't natively line up with its own reference H&E image
(`sw.align_grid`, "Problem A" in this package's design). A multiplex IF scan has no such
ambiguity -- every channel is already a pixel-perfect raster of *itself*, so there is nothing to
self-align. Only the cross-modality step is needed: register the IF scan against Visium HD's own
H&E (`sw.pick_landmarks` + `sw.register_elastic`, "Problem B"), then transfer every channel's
intensity onto the Visium grid via `sw.align(..., registration_result=...)`, exactly like the
MSI notebooks' Step 4/5.

**How the multiplex points are built**: instead of MSI's few hundred thousand acquisition
spots, every pixel of one low-resolution pyramid level of the IF scan becomes one "point", with
all 8 channels as its feature values -- `examples/multiplex_loader.py` reads a single pyramid
level directly from the file's per-channel per-level TIFF pages (no full-resolution decode, no
strided zarr read needed here since this format ships its own precomputed pyramid, unlike the
whole-slide H&E TIFFs used elsewhere in this project).

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

import spatialdata as sd
from spatialdata import SpatialData
from spatialdata.models import Image2DModel
import spatialdata_io
import spatialwarp as sw

from multiplex_loader import read_pyramid_level

/home/croizer/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/croizer/.local/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/croizer/.local/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


## Step 1 -- Visium HD base: full-resolution H&E + scale factor

`spatialdata_io.visium_hd()` needs a `{dataset_id}_feature_slice.h5` to locate the HD layout
metadata, but this particular Space Ranger output isn't dataset-id-prefixed (plain
`feature_slice.h5`), so the symlink fix runs again here. This is only used for the bin
tables/gene counts, though -- the registration reference image comes from
`18H21595_Visium_HE.ndpi` instead of Space Ranger's own downsampled `hires_image`.

**Why the raw NDPI instead of `hires_image`**: the NDPI is a real whole-slide scan (91904 x
119040 px at full resolution, tiled with its own precomputed pyramid), not the ~4600 x 6000 px
render Space Ranger ships -- sharper landmarks, sharper registration. Its filename and
dimensions both point to it being the exact scan this Visium run was calibrated against: the
`scalefactors_json.json`'s `tissue_hires_scalef` (0.050403226) times the NDPI's own full-res
width/height lands within rounding of `hires_image`'s actual size, and the NDPI's own pyramid
levels are exact `/4`, `/16`, `/64` downsamples of that same full-res grid -- so instead of
routing through `hires_scale`/`hires_image` at all, the bin coordinates are rescaled directly
into whichever NDPI pyramid level we read, using that level's own shape ratio to the NDPI's
level-0 shape.

`ndpi_level=2` (5744 x 7440, 16x downsampled, ~0.2s to read) is a good default -- higher detail
than `hires_image` at a similar cost; drop to `3` for a smaller/faster pass if needed.

`bin_size=8` here is just a single-target choice for this tutorial (unlike the multi-resolution
`transfer_msi_to_finer_resolutions.ipynb`) -- swap for `16` or `2` freely, everything downstream
is bin-size-agnostic.

In [2]:
visium_outs = "/home/croizer/Documents/04_Collaborations/06_Pierre-Alexis/outs"
ndpi_path = "/home/croizer/Documents/04_Collaborations/06_Pierre-Alexis/18H21595_Visium_HE.ndpi"
dataset_id = "H21595"
bin_size = 16
bin_size_key = f"square_{bin_size:03d}um"
ndpi_level = 2

feature_slice_link = os.path.join(visium_outs, f"{dataset_id}_feature_slice.h5")
if not os.path.exists(feature_slice_link):
    os.symlink("feature_slice.h5", feature_slice_link)

visium_raw_sdata = spatialdata_io.visium_hd(visium_outs, dataset_id=dataset_id, bin_size=bin_size)

with tifffile.TiffFile(ndpi_path) as tf:
    ndpi_fullres_shape = tf.series[0].shape  # (H, W, 3) at level 0 -- Space Ranger's own fullres grid
    visium_he_array = tf.pages[ndpi_level].asarray()

ndpi_scale = visium_he_array.shape[1] / ndpi_fullres_shape[1]

visium_table = visium_raw_sdata.tables[bin_size_key].copy()
visium_table.obsm["spatial"] = visium_table.obsm["spatial"] * ndpi_scale

visium_sdata = SpatialData(
    images={"he": Image2DModel.parse(np.moveaxis(np.atleast_3d(visium_he_array), -1, 0))},
    tables={"visium": visium_table},
)
print("ndpi_scale:", ndpi_scale, " visium_he_array:", visium_he_array.shape, " bins:", visium_table.n_obs)
visium_sdata

/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/tmp/ipykernel_1104270/3018713610.py:12: UserWarning: No full resolution image found. If incorrect, please specify the path in the `fullres_image_file` parameter when calling the `visium_hd` reader function.
  visium_raw_sdata = spatialdata_io.visium_hd(visium_outs, dataset_id=dataset_id, bin_size=bin_size)


INFO     no axes information specified in the object, setting `dims` to: ('c', 'y', 'x')                           
ndpi_scale: 0.0625  visium_he_array: (5744, 7440, 3)  bins: 170351


/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/.local/lib/python3.10/site-packages/spatialdata/_core/spatialdata.py:185: UserWarning: The table is annotating 'H21595_square_016um', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)


SpatialData object
├── Images
│     └── 'he': DataArray[cyx] (3, 5744, 7440)
└── Tables
      └── 'visium': AnnData (170351, 18085)
with coordinate systems:
    ▸ 'global', with elements:
        he (Images)

## Step 2 -- Load the multiplex IF scan as a point grid

`level` picks which precomputed pyramid level to read -- `4` here is `2^4 = 16x` downsampled from
full resolution (about 2325 x 2967 px, ~6.9M pixels/points); drop to `5` for a faster/coarser
pass, or `3` for finer detail at the cost of ~4x more points. Every pixel of that level becomes
one point in `points_xy`, in that level's own pixel space -- no rescaling needed, since this is
exactly the "points_xy in image's own pixel space" contract `sw.build_spatialdata` expects.

DAPI is used as the registration reference image: it stains every nucleus regardless of channel
panel, so (unlike any single marker channel) it approximates the overall tissue/cell density
pattern an H&E scan would show -- the same role Visium's own H&E plays on the fixed side.

In [3]:
multiplex_tif = "/home/croizer/Documents/04_Collaborations/06_Pierre-Alexis/18H21595.tif"
level = 4

stack, channel_names = read_pyramid_level(multiplex_tif, level=level)
n_channels, height, width = stack.shape
dapi_idx = next(i for i, name in enumerate(channel_names) if "dapi" in name.lower())
composite = stack[dapi_idx]

yy, xx = np.mgrid[0:height, 0:width]
points_xy = np.column_stack([xx.ravel(), yy.ravel()]).astype(float)
values = pd.DataFrame(stack.reshape(n_channels, -1).T, columns=channel_names)

multiplex_sdata = sw.build_spatialdata(composite, points_xy, values=values, image_key="he", table_key="multiplex")
print(f"level {level}: {height} x {width} = {points_xy.shape[0]} points, {n_channels} channels: {channel_names}")
multiplex_sdata

INFO     no axes information specified in the object, setting `dims` to: ('c', 'y', 'x')                           
level 4: 2967 x 2325 = 6898275 points, 8 channels: ['panCK_480 (Opal 480)', 'C31_520 (Opal 520)', 'FAP_570 (Opal 570)', 'CD34_620 (Opal 620)', 'PDPN_690 (Opal 690)', 'aSMA_780 (Opal 780)', 'DAPI (DAPI)', 'AF (Autofluorescence)']


SpatialData object
├── Images
│     └── 'he': DataArray[cyx] (1, 2967, 2325)
└── Tables
      └── 'multiplex': AnnData (6898275, 8)
with coordinate systems:
    ▸ 'global', with elements:
        he (Images)

## Step 3 -- Register the multiplex scan against Visium HD's H&E

Same landmark-picking + elastic registration used for every MSI modality's own H&E -- a window
opens with the DAPI composite (moving, left) and the Visium hires H&E (fixed, right); place a
handful of matching landmarks (tissue corners/edges, distinctive folds) on each side, then close
the window to run `bUnwarpJ`-style elastic registration and save the result.

In [4]:
%matplotlib qt

In [5]:
moving_landmarks, fixed_landmarks, _, _ = sw.pick_landmarks(
    composite, visium_he_array, output_csv=f"landmarks_multiplex_{dataset_id}.csv"
)

registration_result = sw.register_elastic(
    moving_image=composite,
    fixed_image=visium_he_array,
    moving_landmarks=moving_landmarks,
    fixed_landmarks=fixed_landmarks,
    mesh_size=(8, 8),
    number_of_iterations=100,
)
registration_result.save(f"registration_multiplex_{dataset_id}")
registration_result

RegistrationResult(affine_transform=<SimpleITK.SimpleITK.AffineTransform; proxy of <Swig Object of type 'itk::simple::AffineTransform *' at 0x70eb6550b1f0> >, bspline_transform=<SimpleITK.SimpleITK.BSplineTransform; proxy of <Swig Object of type 'itk::simple::BSplineTransform *' at 0x70eb65fdeb30> >, fixed_size=(7440, 5744), correction=None)

## Step 4 -- Transfer all 8 channels onto the Visium HD grid

`distance_threshold` is in Visium HD hires-image pixels, same units/scale as every other
notebook's own threshold -- since the multiplex point grid is dense (one point per pixel of the
chosen level, much finer than a Visium bin), a small threshold is enough for essentially every
bin to find a close match.

In [6]:
merged = sw.align(
    moving=multiplex_sdata,
    fixed=visium_sdata,
    registration_result=registration_result,
    distance_threshold=15.0,
    moving_table_key="multiplex",
    fixed_table_key="visium",
    moving_obsm_key="multiplex",
)
print(f"kept {merged.n_obs} of {visium_sdata.tables['visium'].n_obs} Visium HD bins")
merged

kept 170351 of 170351 Visium HD bins


/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 170351 × 18085
    obs: 'in_tissue', 'array_row', 'array_col', 'location_id', 'region', 'nearest_index', 'nearest_distance'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs', 'spatialwarp'
    obsm: 'spatial', 'spatial_warped', 'multiplex'

In [7]:
merged_before_refine = merged  # keep Step 4's un-refined result around for a before/after comparison

## Step 4.5 -- Fine-tune the registration using a gene-signature match

`KRT7` alone is too sparsely detected at this bin size to be a reliable target for the search
below, so instead of one gene, this builds a small **signature**: the `n_top_genes` genes whose
expression (in the current, pre-refine `merged`) correlates most with the panCK channel
(`"aSMA_780 (Opal 780)"`, per the corrected channel mapping), averaged together into one
per-bin score. Averaging several genes smooths out any single gene's dropout noise while still
tracking the same underlying epithelial signal.

That averaging has to happen over *every* Visium bin, not just the ones Step 4 already matched
-- `sw.refine_alignment` re-matches from scratch as it searches, so bins it newly picks up under
a corrected transform still need a signature value. The correlation step (finding which genes
belong in the signature) still only uses the already-matched `merged`, since it needs paired
gene/channel values; computing each gene's *expression* doesn't need a match at all, so that
part runs against the full `visium_sdata` table directly.

`sw.refine_alignment` then searches the same small rigid nudge (rotation + translation, bounded
by `max_translation`/`max_rotation_deg`) as before, now maximizing correlation against this
signature instead of `KRT7` alone.

In [8]:
import scanpy as sc
import scipy.sparse as sp

if_channel = "CD34_620 (Opal 620)"  # panCK, per the corrected channel mapping
n_top_genes = 20

# -- find genes correlated with the channel, from the already-matched (pre-refine) merged --
adata_norm = merged.copy()
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)

X = adata_norm.X.tocsr() if sp.issparse(adata_norm.X) else sp.csr_matrix(adata_norm.X)
n = X.shape[0]
x = merged.obsm["multiplex"][if_channel].fillna(0).values.astype(float)

mean_g = np.asarray(X.mean(axis=0)).ravel()
var_g = np.asarray(X.multiply(X).sum(axis=0)).ravel() / n - mean_g ** 2
cov_g = (X.T @ x) / n - mean_g * x.mean()
with np.errstate(invalid="ignore", divide="ignore"):
    r = cov_g / np.sqrt(var_g * x.var())

top_idx = np.argsort(-r)[:n_top_genes]
print(f"top {n_top_genes} genes correlated with {if_channel}:")
for gene, r_val in zip(adata_norm.var_names[top_idx], r[top_idx]):
    print(f"  {gene}: r={r_val:.3f}")

# -- average those genes' expression over EVERY bin, not just the already-matched ones.
# Indexed by *position* (top_idx), not by gene name: this dataset's var_names aren't unique
# (same warning seen throughout this notebook), and adata[:, [name, ...]] requires a unique
# index -- merged/visium_sdata's table share the same var order (row-filtering never touches
# columns), so top_idx from the matched `merged` above is valid here unchanged. --
adata_all_norm = visium_sdata.tables["visium"].copy()
sc.pp.normalize_total(adata_all_norm, target_sum=1e4)
sc.pp.log1p(adata_all_norm)
X_all = adata_all_norm.X.tocsc() if sp.issparse(adata_all_norm.X) else np.asarray(adata_all_norm.X)
signature = np.asarray(X_all[:, top_idx].mean(axis=1)).ravel()
visium_sdata.tables["visium"].obs["panck_gene_signature"] = signature

/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/croizer/.local/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:235: UserWarning: Some cells have zero counts
  warn(UserWarning("Some cells have zero counts"))


top 20 genes correlated with CD34_620 (Opal 620):
  MS4A1: r=0.284
  CXCL13: r=0.223
  FDCSP: r=0.215
  CLU: r=0.203
  TNFRSF13C: r=0.186
  CD37: r=0.182
  CD22: r=0.179
  LTF: r=0.178
  BLK: r=0.170
  CXCR4: r=0.165
  LTB: r=0.159
  CD79A: r=0.159
  NIBAN3: r=0.151
  TCL1A: r=0.139
  CD52: r=0.136
  CD19: r=0.135
  SPIB: r=0.132
  SELL: r=0.131
  CD79B: r=0.128
  P2RX5: r=0.124


In [9]:
merged, refined_registration_result = sw.refine_alignment(
    moving=multiplex_sdata,
    fixed=visium_sdata,
    registration_result=registration_result,
    feature_pairs=[("aSMA_780 (Opal 780)", "panck_gene_signature")],
    moving_table_key="multiplex",
    fixed_table_key="visium",
    moving_obsm_key="multiplex",
    distance_threshold=15.0,
    max_translation=30.0,
    max_rotation_deg=5.0,
)
refined_registration_result.save(f"registration_multiplex_{dataset_id}_refined")
print(f"kept {merged.n_obs} of {visium_sdata.tables['visium'].n_obs} Visium HD bins")
print(merged.uns["spatialwarp"]["refinement"])

refine_alignment: fitted correction (rotation=-1.99 deg, tx=30.0, ty=4.5) is at or near its bound -- consider raising max_translation/max_rotation_deg and re-running.
kept 170351 of 170351 Visium HD bins
{'feature_pairs': [('aSMA_780 (Opal 780)', 'panck_gene_signature')], 'rotation_deg': -1.9896424793372471, 'translation': (29.960473133721493, 4.478623733348665), 'correlation': -0.01571139913334341}


/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


### IF-style before/after overlay

Same green/magenta additive-blend, black-background style used later for the all-channels
overlay -- here just the two features that were actually fine-tuned against each other
(`panCK`/`aSMA_780 (Opal 780)` in green, the gene signature in magenta), side by side before
and after Step 4.5. Bright/white patches mean both are strong there; tighter, sharper white
patches after fine-tuning is the visual signature of a genuine improvement -- diffuse green/
magenta fringes around the same regions means it mostly just blurred, not corrected, the match.

In [10]:
from matplotlib.colors import to_rgb

if_overlay_colors = {"channel": "#00FF66", "signature": "#FF00FF"}
vmin_percentile, vmax_percentile = 1, 99

# panck_gene_signature was only added to the full visium_sdata table, not to either merged
# snapshot -- look it up by bin barcode instead of assuming row order/count still line up
signature_by_bin = pd.Series(signature, index=visium_sdata.tables["visium"].obs_names)

def plot_if_overlay(ax, merged_obj, title):
    channel_vals = merged_obj.obsm["multiplex"][if_channel].fillna(0).values
    sig_vals = signature_by_bin.reindex(merged_obj.obs_names).fillna(0).values

    row = merged_obj.obs["array_row"].values.astype(int)
    col = merged_obj.obs["array_col"].values.astype(int)
    row0, col0 = row.min(), col.min()
    n_rows, n_cols = row.max() - row0 + 1, col.max() - col0 + 1

    def to_grid_norm(values):
        vmin, vmax = np.nanpercentile(values, [vmin_percentile, vmax_percentile])
        norm = np.clip((values - vmin) / (vmax - vmin), 0, 1)
        grid = np.zeros((n_rows, n_cols))
        grid[row - row0, col - col0] = norm
        return grid

    rgb = (
        to_grid_norm(channel_vals)[..., None] * np.array(to_rgb(if_overlay_colors["channel"]))
        + to_grid_norm(sig_vals)[..., None] * np.array(to_rgb(if_overlay_colors["signature"]))
    )
    ax.imshow(np.clip(rgb, 0, 1))
    ax.axis("off")
    ax.set_title(title, color="white")

fig, axes = plt.subplots(1, 2, figsize=(18, 9), facecolor="black")
plot_if_overlay(axes[0], merged_before_refine, "Before fine-tuning")
plot_if_overlay(axes[1], merged, "After fine-tuning")
plt.tight_layout()
plt.show()

In [12]:
merged.obsm['multiplex'].columns

Index(['panCK_480 (Opal 480)', 'C31_520 (Opal 520)', 'FAP_570 (Opal 570)',
       'CD34_620 (Opal 620)', 'PDPN_690 (Opal 690)', 'aSMA_780 (Opal 780)',
       'DAPI (DAPI)', 'AF (Autofluorescence)'],
      dtype='object')

In [13]:
channel_rename = {
    "panCK_480 (Opal 480)": "CD21",
    "C31_520 (Opal 520)": "CD3",
    "FAP_570 (Opal 570)": "PNAd",
    "CD34_620 (Opal 620)": "CD20",
    "PDPN_690 (Opal 690)": "CD23",
    "aSMA_780 (Opal 780)": "panCK",
    "DAPI (DAPI)": "DAPI",
    "AF (Autofluorescence)": "AF",
}

merged.obsm["multiplex"] = merged.obsm["multiplex"].rename(columns=channel_rename)
print(merged.obsm["multiplex"].columns)


Index(['CD21', 'CD3', 'PNAd', 'CD20', 'CD23', 'panCK', 'DAPI', 'AF'], dtype='object')


## Step 5 -- QC: a couple of channels on the Visium grid

Visium HD bins sit on a real regular grid, so pivoting into `array_row`/`array_col` and using
`imshow` gives an exact grid with no marker-size guessing -- same plotting recipe used
throughout this project's other Visium HD notebooks.

In [7]:
qc_channels = [channel_names[dapi_idx], "aSMA_780 (Opal 780)"]
vmin_percentile, vmax_percentile = 1, 99

row = merged.obs["array_row"].values.astype(int)
col = merged.obs["array_col"].values.astype(int)
row0, col0 = row.min(), col.min()

fig, axes = plt.subplots(1, len(qc_channels), figsize=(8 * len(qc_channels), 8))
for ax, channel in zip(np.atleast_1d(axes), qc_channels):
    values = merged.obsm["multiplex"][channel].fillna(0).values
    vmin, vmax = np.nanpercentile(values, [vmin_percentile, vmax_percentile])

    grid = np.full((row.max() - row0 + 1, col.max() - col0 + 1), np.nan)
    grid[row - row0, col - col0] = values

    im = ax.imshow(grid, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.axis("off")
    ax.set_title(channel)
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()
plt.show()

In [8]:
gene_name = "KRT7"
if_channel = "aSMA_780 (Opal 780)"  # protein counterpart of ACTA2 -- good sanity check that the two co-localize
vmin_percentile, vmax_percentile = 1, 99

panels = [
    (gene_name, merged.obs_vector(gene_name), "coolwarm"),
    (if_channel, merged.obsm["multiplex"][if_channel].fillna(0).values, "viridis"),
]

row = merged.obs["array_row"].values.astype(int)
col = merged.obs["array_col"].values.astype(int)
row0, col0 = row.min(), col.min()

fig, axes = plt.subplots(1, len(panels), figsize=(8 * len(panels), 8))
for ax, (name, values, cmap) in zip(axes, panels):
    vmin, vmax = np.nanpercentile(values, [vmin_percentile, vmax_percentile])

    grid = np.full((row.max() - row0 + 1, col.max() - col0 + 1), np.nan)
    grid[row - row0, col - col0] = values

    im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.axis("off")
    ax.set_title(name)
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()
plt.show()


In [21]:
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch

channel_colors = {
    #"DAPI (DAPI)": "#0000FF",           # blue -- nuclei
    #"panCK_480 (Opal 480)": "#004CFF",  # green
    "C31_520 (Opal 520)": "#FF00FF",    # magenta
    #"FAP_570 (Opal 570)": "#FFFF00",    # yellow
    "CD34_620 (Opal 620)": "#00FF2A",   # orange
    "PDPN_690 (Opal 690)": "#00FFFF",   # cyan
    "aSMA_780 (Opal 780)": "#FF0000",   # red
   # "AF (Autofluorescence)": "#808080", # grey -- usually background noise; drop this line to exclude it
}

vmin_percentile, vmax_percentile = 1, 99

row = merged.obs["array_row"].values.astype(int)
col = merged.obs["array_col"].values.astype(int)
row0, col0 = row.min(), col.min()
n_rows, n_cols = row.max() - row0 + 1, col.max() - col0 + 1

rgb = np.zeros((n_rows, n_cols, 3))
for channel, hex_color in channel_colors.items():
    values = merged.obsm["multiplex"][channel].fillna(0).values
    vmin, vmax = np.nanpercentile(values, [vmin_percentile, vmax_percentile])
    norm_values = np.clip((values - vmin) / (vmax - vmin), 0, 1)

    grid = np.zeros((n_rows, n_cols))
    grid[row - row0, col - col0] = norm_values

    rgb += grid[..., None] * np.array(to_rgb(hex_color))

rgb = np.clip(rgb, 0, 1)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb)
ax.axis("off")
ax.set_title("All channels overlaid (pseudocolor composite)")

legend_elements = [Patch(facecolor=c, label=ch) for ch, c in channel_colors.items()]
ax.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)

plt.tight_layout()
plt.show()


In [19]:
gene_name = "MS4A1"
if_channel = "CD23"
vmin_percentile, vmax_percentile = 1, 99

row = merged.obs["array_row"].values.astype(int)
col = merged.obs["array_col"].values.astype(int)
row0, col0 = row.min(), col.min()
n_rows, n_cols = row.max() - row0 + 1, col.max() - col0 + 1

def to_grid(values):
    vmin, vmax = np.nanpercentile(values, [vmin_percentile, vmax_percentile])
    norm_values = np.clip((values - vmin) / (vmax - vmin), 0, 1)
    grid = np.zeros((n_rows, n_cols))
    grid[row - row0, col - col0] = norm_values
    return grid

gene_grid = to_grid(merged.obs_vector(gene_name))
if_grid = to_grid(merged.obsm["multiplex"][if_channel].fillna(0).values)

rgb = np.zeros((n_rows, n_cols, 3))
rgb[..., 0] = gene_grid  # red = gene expression
rgb[..., 1] = if_grid    # green = IF channel

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb)
ax.axis("off")
ax.set_title(f"{gene_name} (red) / {if_channel} (green) -- yellow = co-expression")
plt.tight_layout()
plt.show()
